In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
import logging
import torch
from google.colab import drive
from transformers import AutoProcessor, CLIPModel
from PIL import Image

In [14]:
logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger(__name__)

In [12]:
def setup_environment(project_path):
    """
    Mounts Google Drive and changes the current working directory.
    """
    if os.path.exists(project_path):
        os.chdir(project_path)
        logger.info(f"Successfully changed working directory to: {os.getcwd()}")
    else:
        logger.error(f"Project path '{project_path}' does not exist.")

In [5]:
def load_model(model_name="openai/clip-vit-base-patch32"):
    """
    Loads and returns the pre-trained CLIP model and its processor.
    """

    logger.info(f"Loading pre-trained model and processor: {model_name}")
    model = CLIPModel.from_pretrained(model_name)
    processor = AutoProcessor.from_pretrained(model_name)
    logger.info("Model and processor loaded successfully.")

    return model, processor


In [6]:
def extract_image_vectors(images_folder, model, processor):
    """
    Iterates through the folder, extracts features for each image,
    and returns a dictionary with flat lists as values.
    """

    if not os.path.exists(images_folder):
        logger.error(f"Folder '{images_folder}' not found. Aborting processing.")
        return {}

    list_of_images = os.listdir(images_folder)
    image_vectors={}
    logger.info(f"Starting vectorization. Found {len(list_of_images)} files in '{images_folder}'.")

    for image_name in list_of_images:
        image_path = os.path.join(images_folder, image_name)

        try:
            image = Image.open(image_path)
            inputs = processor(images=image, return_tensors="pt", padding=True)

            with torch.inference_mode():
                image_outputs = model.get_image_features(pixel_values=inputs['pixel_values'])

            image_features = image_outputs.pooler_output

            # Flatten the [1, 512] tensor and convert to a standard list
            image_vectors[image_name] = image_features[0].tolist()
            logger.info(f"Successfully processed: {image_name}")

        except Exception as e:
            logger.error(f"Failed to process file '{image_name}'. Error: {e}")

    return image_vectors



In [7]:
def save_to_json(image_vectors, output_path):
    """
    Saves a python dictionary into a clean, indented JSON file.
    """
    with open(output_path, 'w') as f:
        json.dump(image_vectors, f, indent=4)
    logger.info(f"Successfully saved all vectors to file: '{output_path}'")

In [15]:
PROJ_PATH = '/content/drive/My Drive/cvllmproject'

setup_environment(PROJ_PATH)

INFO:__main__:Successfully changed working directory to: /content/drive/My Drive/cvllmproject


In [16]:
images_folder = 'photo'

# Load the DL tools
model, processor = load_model()

INFO:__main__:Loading pre-trained model and processor: openai/clip-vit-base-patch32
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors.index.json "HTTP/1.1 404 No

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: H

In [18]:
# Process the dataset
image_vectors = extract_image_vectors(images_folder, model, processor)

# Save results
if image_vectors:
    save_to_json(image_vectors, 'image_vectors.json')
    logger.info(f"Pipeline completed successfully! Total indexed images: {len(image_vectors)}")
else:
    logger.critical("Pipeline stopped: No images were successfully processed.")

INFO:__main__:Starting vectorization. Found 20 files in 'photo'.
INFO:__main__:Successfully processed: pexels-bingqian-li-230971044-31889972.jpg
INFO:__main__:Successfully processed: pexels-cottonbro-9292784.jpg
INFO:__main__:Successfully processed: pexels-jana-brenner-2161927662-37849928.jpg
INFO:__main__:Successfully processed: pexels-zonghaofeng-28751650.jpg
INFO:__main__:Successfully processed: pexels-sunny67-25033997.jpg
INFO:__main__:Successfully processed: pexels-quang-nguyen-vinh-222549-6875244.jpg
INFO:__main__:Successfully processed: pexels-krisof-2674905.jpg
INFO:__main__:Successfully processed: pexels-szafran-33153446.jpg
INFO:__main__:Successfully processed: pexels-724211268-33585483.jpg
INFO:__main__:Successfully processed: pexels-denniz-futalan-339724-6716838.jpg
INFO:__main__:Successfully processed: pexels-taryn-elliott-4440220.jpg
INFO:__main__:Successfully processed: pexels-cannontaler-15093520.jpg
INFO:__main__:Successfully processed: pexels-arnaud-38219370.jpg
INFO: